# Exercise 05

## Demographic Prisoner's Dilemma

The Demographic Prisoner's Dilemma is a family of variants on the classic two-player [Prisoner's Dilemma](https://en.wikipedia.org/wiki/Prisoner's_dilemma), first developed by [Joshua Epstein](https://doi.org/10.1002/(SICI)1099-0526(199811/12)4:2%3C36::AID-CPLX9%3E3.0.CO;2-Z   ). The model consists of agents, each with a strategy of either Cooperate or Defect, playing against each neighbour in the Moore-neighbourhood. Each agent's payoff is based on its strategy and the strategies of its spatial neighbors. After each step of the model, the agents adopt the strategy of their neighbor (or their own) with the highest total score. 

The specific variant presented here is adapted from the [Evolutionary Prisoner's Dilemma](http://ccl.northwestern.edu/netlogo/models/Prisoner'sDilemmaBasicEvolutionary) model included with NetLogo. Its payoff table is a slight variant of the traditional PD payoff table:

<table>
    <tr><td></td><td>Cooperate</td><td>Defect</td></tr>
    <tr><td>Cooperate</td><td>1, 1</td><td>0, D</td></tr>
    <tr><td>Defect</td><td>D, 0</td><td>0, 0</td></tr>
</table>

Where *D* is the defection bonus, generally set higher than 1. In these runs, the defection bonus is set to $D=1.6$.

The Demographic Prisoner's Dilemma demonstrates how simple rules can lead to the emergence of widespread cooperation, despite the Defection strategy dominating each individual interaction game. However, it is also interesting for another reason: it is known to be sensitive to the activation regime employed in it.

Below, we demonstrate this by instantiating the same model (with the same random seed) three times, with three different activation regimes: 

* Sequential activation, where agents are activated in the order they were added to the model;
* Random activation, where they are activated in random order every step;
* Simultaneous activation, simulating them all being activated simultaneously.



In [ ]:
import sys

sys.path.insert(0,'./')
from pd_grid.model import PdGrid, PrisonersDilemmaScenario

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec
from matplotlib.colors import LinearSegmentedColormap
from mesa.visualization import show_logs, freeze_logs

%matplotlib inline
matplotlib.rcParams['font.size'] = 12

## Helper functions

In [ ]:
uniks = LinearSegmentedColormap.from_list( 'unik', [np.array((80,149,200))/255, np.array((74,172,150))/255,
                                                  np.array((234,195,114))/255, np.array((199,16,92))/255])

def draw_grid(model, ax=None):
    """
    Draw the current state of the grid, with Defecting agents in red
    and Cooperating agents in blue.
    """
    if not ax:
        fig, ax = plt.subplots(figsize=(6, 6))
    grid = np.zeros((model.grid.width, model.grid.height))
    for cell in model.grid.all_cells:
        if cell.agents[0].move == "D":
            grid[cell.coordinate[1]][cell.coordinate[0]] = 1
        else:
            grid[cell.coordinate[1]][cell.coordinate[0]] = 0
    ax.pcolormesh(grid, cmap=uniks, vmin=0, vmax=1)
    ax.axis("off")
    ax.set_title("Steps: {}".format(model.time))

In [ ]:
def draw_score_grid(model, ax=None, minvalue=None, maxvalue=None):
    """
    Draw the current state of the grid, with Defecting agents in red
    and Cooperating agents in blue.
    """
    if maxvalue == None:
        maxvalue = max(model.agents, key=lambda agent: agent.score).score
    if minvalue == None:
        minvalue = min(model.agents, key=lambda agent: agent.score).score
    if not ax:
        fig, ax = plt.subplots(figsize=(6, 6))
    grid = np.zeros((model.grid.width, model.grid.height))
    for cell in model.grid.all_cells:
        grid[cell.coordinate[1]][cell.coordinate[0]] = cell.agents[0].score
    fig = ax.pcolormesh(grid, vmin=minvalue, vmax=maxvalue)
    ax.axis("off")
    plt.colorbar(fig, location="bottom")
    ax.set_title("Step: {}".format(model.time))

In [ ]:
def run_model(model, steps = [10,10,20,20,40], maxscorevalue = None):
    """
    Run an experiment with a given model, and plot the results.
    """
    fig = plt.figure(figsize=(12, 6))
    gs = gridspec.GridSpec(3, len(steps) + 1, height_ratios=[2, 3, 3]) 
    for i in range(0, len(steps)):
        draw_grid(model, fig.add_subplot(gs[i]))
        draw_score_grid(model, fig.add_subplot(gs[i + 1 + len(steps)]),
                        maxvalue=maxscorevalue)
        model.run_for(steps[i])
        
    draw_grid(model, fig.add_subplot(gs[i+1]))
    draw_score_grid(model, fig.add_subplot(gs[i + 2 + len(steps)]),
                    maxvalue=maxscorevalue)
    
    ax = fig.add_subplot(3, 1, 3)

    d = model.datacollector.get_model_vars_dataframe()
    d = d * 100 / len(model.agents)
    d.plot(ax=ax, colormap=uniks)
    return model

In [ ]:
def compare_runs(models):
    data = pd.DataFrame()
    for model in models:
        modeldata = model.datacollector.get_model_vars_dataframe() * 100 / len(model.agents)
        modeldata.columns = [model.activation_order]
        data = pd.concat([data, modeldata],axis=1)
        
    fig = plt.figure(figsize=(12, 8))
    ax1 = fig.add_subplot(111)
    data.plot(ax=ax1, colormap=uniks)
    fig.savefig("dpd_compare.png")

## Subtask 2.2

Explain the different evolution of cooperators for random and simultaneous activation schemes by exploring the warm-up phase. Why does for some random seeds the number of cooperators go down before it raises, but not for sequential? (< 300 words)

#### Run DPD in browser

In [ ]:
from mesa.visualization import SolaraViz
from pd_grid.app import model_params, plot_component, renderer

# Initialize model
scenario = PrisonersDilemmaScenario(
    rng=25,
    width = 30,
    height = 30,
    activation_order =  "Random",
)
initial_model = PdGrid(scenario=scenario)

page = SolaraViz(
    model=initial_model,
    renderer=renderer,
    components=[plot_component],
    model_params=model_params,
    name="Spatial Prisoner's Dilemma",
)
page  

### Warm-up: Random activation

Passing `printneighbourscore=True` to model initialisation prints score and `printneighbourorder=True` prints move information about the focal (passing `focalpos=(0,0)`) agents' neighbours. The order of neighbours is row-wise (eg. starting with upper left, then upper middle, upper right, middle left aso.)

In [ ]:
show_logs("DPD")
scenario = PrisonersDilemmaScenario(
    rng=25,
    width = 10,
    height = 10,
    activation_order =  "Random",
)
m_random = PdGrid(scenario=scenario,
                  focalpos=(0,0),
                  printneighbourscore=False,
                  printneighbourorder=False)
run_model(m_random, steps = [1,1,1,1,1,10])
freeze_logs("DPD")

### Warm-up: Simultaneous Activation

In [ ]:
show_logs("DPD")
scenario = PrisonersDilemmaScenario(
    rng=25,
    width = 10,
    height = 10,
    activation_order =  "Simultaneous",
)
m_simultaneous = PdGrid(scenario=scenario,
                  focalpos=(0,0),
                  printneighbourscore=False,
                  printneighbourorder=False)
run_model(m_simultaneous, steps = [1,1,1,1,1,1])
freeze_logs("DPD")

### Compare runs

In [ ]:
matplotlib.rcParams['font.size'] = 12
compare_runs([m_random, m_simultaneous])

## Subtask 2.3

**Discuss the different activation schemes in light of possible model purposes in general and in particular to the DPD. Which one would you chose for which purpose (give examples), and which not? (<200 words)?**

## Subtask 3.1
**Make yourself familiar with running batch runs with the Evacuation model (also remember the [mesa tutorial](https://mesa.readthedocs.io/stable/tutorials/0_first_model.html#running-your-first-model)). Why is the `replications` parameter important?**

## Subtask 3.2

**Conduct batch runs with variations in `cooperation_mean` and `nervousness_mean` and analyse and discuss the results by adopting the provided code in the jupyter notebook. What effect has an increase of the replication parameter, e.g. to `100`? Which important statistics formula is relevant (<200 words)?**

In [ ]:
from mesa.visualization import show_logs, freeze_logs
from mesa.visualization import SolaraViz, SpaceRenderer
import matplotlib.pyplot as plt
from matplotlib.colorizer import Colorizer
import pandas as pd
import numpy as np
import sys
import time
from datetime import timedelta
import itertools
sys.path.insert(0,'../../abmodel')

from fire_evacuation.model import FireEvacuation,FireEvacuationScenario
from fire_evacuation.app import agent_portrayal, model_params, make_plot_component
from fire_evacuation.agent import Human

In [ ]:
# In case you want to explore the model via GUI:
scenario = FireEvacuationScenario(
        random_spawn = True,
        floor_size = 14,
        human_count = 100,
        alarm_believers_prop = 0.2,
        facilitators_percentage = 5,
        panic_rush=False,
        max_steps = 200,
        max_speed = 2,
        seed = 3
)
model = FireEvacuation(scenario)
renderer = SpaceRenderer(model, backend="matplotlib").setup_agents(agent_portrayal)
renderer.render()
page = SolaraViz(
    model,
    model_params = model_params,
    renderer=renderer,
    name="Evacuation Model",
    components=[make_plot_component("AvgNervousness"),],
)
page  # noqa

Note that the simulation may take a few minutes to terminate!

In [ ]:
def batchrun_model(
        scenario,
        experiments,
        replications=20,
        aggregate_output_per_run=True,
    ):
    """
    Instantiates Scenarios for parameter/replication variations, runs the model
    for every Scenario and concats results in a pd.DataFrame.

    Parameters
    ----------
    experiments: pd.DataFrame or dict
        data frame of parameter combinations with parameter names as column names
    replications: int
        number of runs with different RNG
    max_step: int
        the number of steps the model is run in case it does not terminate earlier
    """
    # unfortunately, scenario.from_dataframe() is a class method and does not consider values in scenario,
    # so we need to add them to experiments manually:
    experiments = {key: [value] for key, value in scenario.__dict__.items()} | experiments
    
    # create combinations of given parameter values
    if type(experiments) is pd.DataFrame:
        experiments = pd.DataFrame(itertools.product(*[experiments[col] for col in experiments.columns]),
                               columns=experiments.columns)
    elif type(experiments) is dict:
        experiments = pd.DataFrame(itertools.product(*experiments.values()),
                               columns=experiments.keys())
    
    scenarios = scenario.from_dataframe(
            experiments=experiments,
            replications=replications,
    )
    results = pd.DataFrame()
    
    print(f"Perform {len(scenarios)} model runs...")
    start = time.time()
    for scenario in scenarios:
        model =  FireEvacuation(scenario=scenario)
        model.run_model()
        d = model.datacollector.get_model_vars_dataframe()
        d.insert(0, "sid", scenario.scenario_id)
        d.insert(0, "rid", scenario.replication_id)
        d.index.name="Step"
        d = d.reset_index()
        # calc means
        if aggregate_output_per_run:
            d = d.groupby(["sid", "rid"]).agg(
                {column:"mean" if column != "Step" else "max" for column in d.columns}
            )
        d = pd.concat([pd.DataFrame(scenario.__dict__, index=d.index), d], axis=1)
        results = pd.concat([results, d])
    print(f"Done! Took {str(timedelta(seconds=time.time() - start))} h")
    return results

In [ ]:
results = batchrun_model(
    scenario = FireEvacuationScenario(
        random_spawn = True,
        floor_size = 14,
        human_count = 100,
        alarm_believers_prop = 0.2,
        facilitators_percentage = 5,
        panic_rush=False,
        max_speed = 2,
        max_steps = 200,
    ),
    experiments =
        {"cooperation_mean": np.arange(0.1,1.0,0.2),
        "nervousness_mean": np.arange(0.1,1.0,0.2),
        },
    replications=5,
    aggregate_output_per_run=True,
)

In [ ]:
# visualiation: Duration of Escape
data = results[['cooperation_mean', 'nervousness_mean','Step']].round(decimals=1)
df = data.groupby(['cooperation_mean', 'nervousness_mean']).agg("mean")
plot_df = (df.reset_index()
              .pivot(index='cooperation_mean', columns='nervousness_mean'))

fig, ax = plt.subplots()
im = ax.imshow(plot_df, colorizer=Colorizer(cmap="RdYlGn_r"))
plt.colorbar(im)
plt.xlabel('nervousness_mean')
plt.ylabel('cooperation_mean')
plt.title("Duration of escape")
ax.set_xticks(np.arange(plot_df.shape[1]))
ax.set_xticklabels(sorted(set(round(data['nervousness_mean'], ndigits=1))))
ax.set_yticks(np.arange(plot_df.shape[0]))
ax.set_yticklabels(plot_df.index)
None

In [ ]:
# visualiation: Number of Helped
data = results[['cooperation_mean', 'nervousness_mean','NumHelping']].round(decimals=1)
df = data.groupby(['cooperation_mean', 'nervousness_mean']).agg("mean")
plot_df = (df.reset_index()
              .pivot(index='cooperation_mean', columns='nervousness_mean'))

fig, ax = plt.subplots()
im = ax.imshow(plot_df, colorizer=Colorizer(cmap="RdYlGn"))
plt.colorbar(im)
plt.xlabel('nervousness_mean')
plt.ylabel('cooperation_mean')
plt.title("Number of helped humans")
ax.set_xticks(np.arange(plot_df.shape[1]))
ax.set_xticklabels(sorted(set(round(data['nervousness_mean'], ndigits=1))))
ax.set_yticks(np.arange(plot_df.shape[0]))
ax.set_yticklabels(plot_df.index)
None

## Subtask 3.3

**Formulate a plausible numerical definition of a panic during the evacuation. Consider the proportion of escaped individuals and time. Can you change the output of batch_run to check your criteria (you don’t have t implement this, but your’re free to do so)?**

## Subtask 3.4

**Discuss whether your notion of panic fulfils the criteria for an emergent phenomenon or whether it is imposed.**